In [13]:
import json

from minsearch import Index
from anthropic import Anthropic
from gitsource import GithubRepositoryDataReader

In [7]:
client = Anthropic()

In [ ]:
# Retrieve the data from the GitHub repository
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
documents = [f.parse() for f in files]

print(f"Loaded {len(documents)} documents")

Loaded 95 documents


In [14]:
# Create an index for the documents
chunks_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunks_index.fit(documents)

In [ ]:
# Search the index for relevant documents based on a query
query = "How do I use Evidently to monitor my ML model in production?"
results = chunks_index.search(query)

In [18]:
search_result_json = json.dumps(results, indent=2)

In [19]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT.
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [53]:
def llm(
    prompt: str,
    instructions: str | None = None,
    model: str = "claude-opus-5",
    max_tokens: int = 2056,
    ) -> str:
    response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {"role": "assistant", "content": instructions or "You are a helpful course assistant."},
            {"role": "user", "content": prompt}
        ],
        thinking={"type": "adaptive", "display": "summarized"},
    )
    for block in response.content:
        if block.type == "text":
            return block.text

In [55]:
def build_prompt(query: str, search_results: list[dict]) -> str:
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [56]:
def search(query):
    return chunks_index.search(query, num_results=5)

In [57]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [58]:
rag('how do I implement llm as a judge?')

'## Implementing an LLM-as-a-Judge with Evidently\n\nAn LLM judge is essentially a small ML system you build with a prompt — so the workflow is: **create an eval dataset → design the judge prompt → run it over your data → evaluate the judge itself**.\n\n---\n\n### 1. Install and import\n\n```python\npip install evidently\n```\n\n```python\nimport pandas as pd\nfrom evidently import Dataset, DataDefinition, Report, BinaryClassification\nfrom evidently.descriptors import *\nfrom evidently.presets import TextEvals, ValueStats, ClassificationPreset\nfrom evidently.llm.templates import BinaryClassificationPromptTemplate\n\nimport os\nos.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n```\n\n(You can swap in other evaluator LLMs — see the "Change the evaluator LLM" docs.)\n\n---\n\n### 2. Prepare an evaluation dataset\n\nBuild a small dataset with your inputs, outputs, and — importantly — **your own manual labels + comments**. Labeling first helps you (a) formulate better criteria and (b) have groun